# 03 · Load
**Purpose:** Write the cleaned data into a SQLite database using idempotent upserts.  
SQLite needs no server — perfect for Codespaces. Swap to PostgreSQL for production.

---
### Key design decisions
| Decision | Rationale |
|---|---|
| `INSERT OR IGNORE` | Safe to re-run — no duplicate rows on `(coin_id, ingested_at)` |
| SQLite default | Zero config, ships with Python, works in any Codespace |
| `pipeline_run_log` table | Full audit trail for every ETL run |
| Indexes on `coin_id` + `ingested_at` | Fast time-series and per-coin queries |


In [ ]:
import os, sqlite3, logging
from datetime import datetime, timezone
from contextlib import contextmanager

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger(__name__)

SQLITE_PATH = "data/crypto_market.db"
os.makedirs("data", exist_ok=True)
print(f"✓ DB path: {os.path.abspath(SQLITE_PATH)}")


---
## Step 1 — Connection helper


In [ ]:
@contextmanager
def get_connection(path=SQLITE_PATH):
    """Context manager for SQLite — commits on success, rolls back on error."""
    conn = sqlite3.connect(path)
    conn.row_factory = sqlite3.Row
    try:
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

print("✓ Connection helper defined")


---
## Step 2 — Schema bootstrap
Creates all tables and indexes if they don't already exist.


In [ ]:
CREATE_COIN_PRICES_SQL = """
CREATE TABLE IF NOT EXISTS coin_prices (
    id                          INTEGER PRIMARY KEY AUTOINCREMENT,
    coin_id                     TEXT NOT NULL,
    symbol                      TEXT NOT NULL,
    name                        TEXT NOT NULL,
    current_price_usd           REAL,
    market_cap_usd              INTEGER,
    market_cap_rank             INTEGER,
    fully_diluted_valuation_usd INTEGER,
    total_volume_24h_usd        INTEGER,
    high_24h_usd                REAL,
    low_24h_usd                 REAL,
    price_change_24h_usd        REAL,
    price_change_pct_1h         REAL,
    price_change_pct_24h        REAL,
    price_change_pct_7d         REAL,
    volatility_score            REAL,
    circulating_supply          REAL,
    total_supply                REAL,
    max_supply                  REAL,
    ath_usd                     REAL,
    ath_change_pct              REAL,
    atl_usd                     REAL,
    last_updated                TEXT,
    ingested_at                 TEXT NOT NULL,
    UNIQUE(coin_id, ingested_at)
)
"""

CREATE_GLOBAL_STATS_SQL = """
CREATE TABLE IF NOT EXISTS global_market_stats (
    id                          INTEGER PRIMARY KEY AUTOINCREMENT,
    total_market_cap_usd        INTEGER,
    total_volume_24h_usd        INTEGER,
    btc_dominance_pct           REAL,
    eth_dominance_pct           REAL,
    active_cryptocurrencies     INTEGER,
    total_exchanges             INTEGER,
    market_cap_change_pct_24h   REAL,
    ingested_at                 TEXT NOT NULL,
    UNIQUE(ingested_at)
)
"""

CREATE_RUN_LOG_SQL = """
CREATE TABLE IF NOT EXISTS pipeline_run_log (
    id              INTEGER PRIMARY KEY AUTOINCREMENT,
    run_id          TEXT NOT NULL,
    started_at      TEXT NOT NULL,
    completed_at    TEXT,
    records_loaded  INTEGER DEFAULT 0,
    status          TEXT DEFAULT 'RUNNING',
    error_message   TEXT
)
"""

INDEXES_SQL = [
    "CREATE INDEX IF NOT EXISTS idx_coin_prices_coin_id ON coin_prices(coin_id)",
    "CREATE INDEX IF NOT EXISTS idx_coin_prices_ingested_at ON coin_prices(ingested_at)",
    "CREATE INDEX IF NOT EXISTS idx_global_stats_ingested_at ON global_market_stats(ingested_at)",
]

def bootstrap_schema():
    """Create all tables and indexes."""
    with get_connection() as conn:
        cur = conn.cursor()
        cur.execute(CREATE_COIN_PRICES_SQL)
        cur.execute(CREATE_GLOBAL_STATS_SQL)
        cur.execute(CREATE_RUN_LOG_SQL)
        for idx in INDEXES_SQL:
            cur.execute(idx)
    logger.info("✓ Schema bootstrap complete")

bootstrap_schema()


---
## Step 3 — Load coin prices


In [ ]:
def load_coin_prices(records: list) -> int:
    """Upsert a batch of transformed coin price records. Returns rows loaded."""
    if not records:
        return 0

    cols = list(records[0].keys())
    placeholders = ", ".join(["?" for _ in cols])
    col_names    = ", ".join(cols)
    sql = f"INSERT OR IGNORE INTO coin_prices ({col_names}) VALUES ({placeholders})"

    with get_connection() as conn:
        cur = conn.cursor()
        rows = [tuple(r[c] for c in cols) for r in records]
        cur.executemany(sql, rows)
        loaded = cur.rowcount if cur.rowcount >= 0 else len(records)

    logger.info(f"✓ Loaded {loaded} coin price records")
    return loaded

print("✓ load_coin_prices defined")


In [ ]:
# ── Pull fresh data and load ─────────────────────────────────────
import requests
from datetime import datetime, timezone

BASE_URL = "https://api.coingecko.com/api/v3"
COINS = ["bitcoin","ethereum","solana","cardano","polkadot","chainlink","avalanche-2","uniswap"]

def fetch_and_transform():
    params = {
        "vs_currency": "usd", "ids": ",".join(COINS),
        "order": "market_cap_desc", "per_page": len(COINS), "page": 1,
        "sparkline": "false", "price_change_percentage": "1h,24h,7d"
    }
    raw = requests.get(f"{BASE_URL}/coins/markets", params=params, timeout=15).json()
    ingested_at = datetime.now(timezone.utc).isoformat()
    cleaned = []
    for r in raw:
        p1h  = r.get("price_change_percentage_1h_in_currency")  or 0.0
        p24h = r.get("price_change_percentage_24h_in_currency") or 0.0
        p7d  = r.get("price_change_percentage_7d_in_currency")  or 0.0
        lu   = r.get("last_updated", ingested_at)
        if lu:
            lu = datetime.fromisoformat(lu.replace("Z", "+00:00")).isoformat()
        cleaned.append({
            "coin_id": r.get("id","unknown"), "symbol": (r.get("symbol") or "").upper(),
            "name": r.get("name","Unknown"),
            "current_price_usd": round(float(r.get("current_price") or 0), 6),
            "market_cap_usd": int(r.get("market_cap") or 0),
            "market_cap_rank": r.get("market_cap_rank"),
            "fully_diluted_valuation_usd": r.get("fully_diluted_valuation"),
            "total_volume_24h_usd": int(r.get("total_volume") or 0),
            "high_24h_usd": round(float(r.get("high_24h") or 0), 6),
            "low_24h_usd": round(float(r.get("low_24h") or 0), 6),
            "price_change_24h_usd": round(float(r.get("price_change_24h") or 0), 6),
            "price_change_pct_1h": round(p1h, 4), "price_change_pct_24h": round(p24h, 4),
            "price_change_pct_7d": round(p7d, 4),
            "volatility_score": round(abs(p1h - p24h), 4),
            "circulating_supply": r.get("circulating_supply"),
            "total_supply": r.get("total_supply"), "max_supply": r.get("max_supply"),
            "ath_usd": round(float(r.get("ath") or 0), 6),
            "ath_change_pct": round(float(r.get("ath_change_percentage") or 0), 4),
            "atl_usd": round(float(r.get("atl") or 0), 10),
            "last_updated": lu, "ingested_at": ingested_at,
        })
    return cleaned

clean_market_data = fetch_and_transform()
n_loaded = load_coin_prices(clean_market_data)
print(f"Loaded {n_loaded} records into coin_prices")


---
## Step 4 — Load global stats


In [ ]:
def load_global_stats(stats: dict) -> int:
    """Upsert one global market stats record."""
    if not stats:
        return 0
    cols = list(stats.keys())
    placeholders = ", ".join(["?" for _ in cols])
    sql = f"INSERT OR IGNORE INTO global_market_stats ({', '.join(cols)}) VALUES ({placeholders})"
    with get_connection() as conn:
        cur = conn.cursor()
        cur.execute(sql, tuple(stats[c] for c in cols))
    logger.info("✓ Loaded global market stats")
    return 1

def fetch_global():
    r = requests.get(f"{BASE_URL}/global", timeout=15)
    r.raise_for_status()
    raw = r.json().get("data", {})
    mc  = raw.get("total_market_cap", {})
    vol = raw.get("total_volume", {})
    pct = raw.get("market_cap_percentage", {})
    return {
        "total_market_cap_usd":      int(mc.get("usd") or 0),
        "total_volume_24h_usd":      int(vol.get("usd") or 0),
        "btc_dominance_pct":         round(float(pct.get("btc") or 0), 4),
        "eth_dominance_pct":         round(float(pct.get("eth") or 0), 4),
        "active_cryptocurrencies":   raw.get("active_cryptocurrencies"),
        "total_exchanges":           raw.get("markets"),
        "market_cap_change_pct_24h": round(float(raw.get("market_cap_change_percentage_24h_usd") or 0), 4),
        "ingested_at":               datetime.now(timezone.utc).isoformat(),
    }

clean_global = fetch_global()
load_global_stats(clean_global)


---
## Step 5 — Audit log


In [ ]:
def log_run(run_id: str, status: str, records_loaded: int = 0, error: str = None):
    """Write a pipeline run entry to the audit log."""
    now = datetime.now(timezone.utc).isoformat()
    with get_connection() as conn:
        conn.execute(
            """INSERT INTO pipeline_run_log
               (run_id, started_at, completed_at, records_loaded, status, error_message)
               VALUES (?, ?, ?, ?, ?, ?)""",
            (run_id, now, now, records_loaded, status, error)
        )
    logger.info(f"Run log written: {run_id} → {status}")

import uuid
run_id = str(uuid.uuid4())[:8]
log_run(run_id, "SUCCESS", records_loaded=n_loaded)
print(f"Run logged: {run_id}")


---
## Step 6 — Query the database


In [ ]:
# ── Latest price snapshot ────────────────────────────────────────
with get_connection() as conn:
    cur = conn.cursor()
    rows = cur.execute("""
        SELECT coin_id, symbol, current_price_usd, price_change_pct_24h,
               volatility_score, market_cap_rank
        FROM coin_prices
        WHERE ingested_at = (SELECT MAX(ingested_at) FROM coin_prices)
        ORDER BY market_cap_rank
    """).fetchall()

print(f"{'#':<4} {'Coin':<12} {'Price (USD)':>14} {'24h %':>8} {'Volatility':>12}")
print("-" * 55)
for r in rows:
    arrow = "▲" if r['price_change_pct_24h'] >= 0 else "▼"
    print(f"  {r['market_cap_rank']:<3} {r['coin_id']:<12} ${r['current_price_usd']:>13,.4f} "
          f"{arrow}{abs(r['price_change_pct_24h']):>6.2f}% {r['volatility_score']:>12.4f}")


In [ ]:
# ── Most volatile coins ──────────────────────────────────────────
with get_connection() as conn:
    rows = conn.execute("""
        SELECT coin_id, symbol, volatility_score, price_change_pct_1h, price_change_pct_24h
        FROM coin_prices
        WHERE ingested_at = (SELECT MAX(ingested_at) FROM coin_prices)
        ORDER BY volatility_score DESC
        LIMIT 5
    """).fetchall()

print("Most volatile coins (highest 1h vs 24h divergence):")
print("-" * 55)
for r in rows:
    print(f"  {r['coin_id']:<15} score={r['volatility_score']:.4f}  "
          f"1h={r['price_change_pct_1h']:+.3f}%  24h={r['price_change_pct_24h']:+.3f}%")


In [ ]:
# ── Pipeline run log ─────────────────────────────────────────────
with get_connection() as conn:
    rows = conn.execute("""
        SELECT run_id, status, records_loaded, started_at
        FROM pipeline_run_log
        ORDER BY started_at DESC LIMIT 10
    """).fetchall()

print(f"{'Run ID':<10} {'Status':<10} {'Records':>8}  Started at")
print("-" * 55)
for r in rows:
    print(f"  {r['run_id']:<8} {r['status']:<10} {r['records_loaded']:>8}  {r['started_at']}")


---
## ✅ Load complete

Data is now in `data/crypto_market.db`.  
See **04_pipeline_end_to_end.ipynb** to run the full ETL in one shot,
or open the DB file directly with any SQLite viewer.
